# What Makes an LLM an Agent

A raw **large language model (LLM)** is a text predictor: given tokens, it predicts the next token. That is powerful for drafting, explaining, and classifying — but by itself the model **cannot** check a live ticket queue, look up a runbook, or close a support case.

An **LLM agent** wraps the same predictor inside a **runtime** that adds:

- **Tools** — functions the model can invoke to change or read the world.
- **Memory** — more than the current chat window (working, episodic, semantic).
- **Planning** — deciding what to do next, step by step.
- **Feedback** — learning from tool results, errors, critics, or humans.
- **Stop rules & guardrails** — when to finish and what is forbidden.

This notebook answers one practical question: *How do you tell whether a feature is "just an LLM" or a genuine agent?* You will walk the **autonomy spectrum**, see **tool use as agency**, compare **memory and planning** patterns, and apply a **6-point agent checklist** to real product ideas.

---

## Scenario: IT Support Ticket Assistant

Throughout this notebook we use a fictional **IT Support Ticket Assistant** that helps employees with password resets, VPN issues, and access requests. The same user question — *"I cannot connect to VPN from home"* — is handled at four different autonomy levels so you can see exactly what changes when an LLM becomes an agent.

Everything is self-contained below. No prior notebooks or external corpora are required.

In [ ]:
"""
Shared environment for the IT Support Ticket Assistant.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

# --- Semantic memory: internal runbooks (knowledge the agent can retrieve) ---
RUNBOOKS: list[dict[str, str]] = [
    {
        "id": "vpn-101",
        "title": "VPN connection troubleshooting",
        "text": (
            "1. Confirm corporate Wi-Fi or wired network is not blocking UDP 443. "
            "2. Restart the VPN client and clear cached credentials. "
            "3. Verify MFA device is synced. "
            "4. If error persists, collect client logs and open ticket category NETWORK."
        ),
    },
    {
        "id": "pwd-102",
        "title": "Password reset procedure",
        "text": (
            "Direct users to https://sso.company.internal/reset. "
            "After reset, wait 5 minutes before retrying VPN. "
            "Escalate to Identity team if account is locked."
        ),
    },
    {
        "id": "access-103",
        "title": "Software access requests",
        "text": (
            "Create an access request ticket with manager approval. "
            "Standard provisioning SLA is 2 business days. "
            "Do not share admin credentials under any circumstance."
        ),
    },
]

# --- Working + episodic state: ticket queue (simulated service desk API) ---
TICKET_QUEUE: list[dict[str, Any]] = [
    {
        "id": "INC-9001",
        "employee": "alice@company.com",
        "subject": "VPN drops every 10 minutes",
        "status": "open",
        "category": "NETWORK",
        "created_at": "2026-07-08T11:00:00Z",
    },
]

# Episodic memory: summaries of past resolved sessions (simulates long-term store)
EPISODIC_MEMORY: list[dict[str, str]] = [
    {
        "session_id": "sess-772",
        "summary": "Resolved VPN issue for bob@company.com by clearing cached credentials (vpn-101 step 2).",
    },
]

print(f"Runbooks loaded: {len(RUNBOOKS)}")
print(f"Open tickets: {sum(1 for t in TICKET_QUEUE if t['status'] == 'open')}")
print(f"Past sessions in episodic memory: {len(EPISODIC_MEMORY)}")

---

## 1. LLM as Predictor vs LLM as Agent

| Mode | What goes in | What comes out | Changes the environment? |
|------|--------------|----------------|--------------------------|
| **Predictor** | A prompt | Text completion | No |
| **Tool caller** | Prompt + tool schemas | Structured call + optional text | Yes — *your code* runs the tool |
| **Agent** | Goal + history + tools + stop rules | Repeated decisions until done | Yes — across multiple steps |

```
Predictor:  prompt ──► LLM ──► answer (done)

Agent:      goal ──► LLM ──► tool call? ──► execute ──► observation ──► LLM ──► ... ──► final answer
```

The critical insight: **the LLM never touches your database or API directly.** It proposes actions; the **agent runtime** validates, executes, and feeds results back. Agency lives in the **loop around the model**, not inside the weights.

---

## 2. The Autonomy Spectrum

Autonomy is not binary. The same product can sit at different levels depending on how much decision-making you give the model:

```
LOW AUTONOMY                                              HIGH AUTONOMY
│                                                                 │
▼                                                                 ▼
fixed prompt    RAG + single call    ReAct tool loop    multi-role system
template        retrieve + answer     search → act       planner + specialists
```

| Level | Name | What the system does | VPN example |
|-------|------|----------------------|-------------|
| **L0** | Static prompt | Hard-coded context in the system message | "Here is the VPN runbook: …" pasted into prompt |
| **L1** | Retrieve + answer | One retrieval step, one LLM call | Search runbooks for "VPN", compose answer |
| **L2** | Tool loop (ReAct) | Model chooses tools across multiple steps | Search runbooks → check open tickets → create ticket |
| **L3** | Multi-role | Planner delegates to specialists | Triage agent → network specialist → ticket closer |

Higher autonomy can solve harder problems — but adds latency, cost, and failure modes. Choose the **lowest level that works**.

In [ ]:
USER_QUESTION = "I cannot connect to VPN from home. What should I do?"


def level_0_static_prompt(question: str) -> str:
    """L0: All context baked into the prompt — no retrieval, no tools."""
    vpn_text = RUNBOOKS[0]["text"]
    return (
        f"[L0 Static] Based on the embedded runbook: {vpn_text} "
        f"(Question was: {question})"
    )


def search_runbooks(query: str, top_k: int = 2) -> list[dict[str, str]]:
    """Keyword search over runbooks — used by L1 and L2."""
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for rb in RUNBOOKS:
        haystack = f"{rb['title']} {rb['text']}".lower()
        score = sum(1 for term in terms if term in haystack)
        if score:
            scored.append((score, rb))
    scored.sort(key=lambda x: (-x[0], x[1]["id"]))
    return [{"id": r["id"], "title": r["title"], "text": r["text"]} for _, r in scored[:top_k]]


def level_1_retrieve_and_answer(question: str) -> str:
    """L1: One retrieval step, then compose answer — still a single pass."""
    hits = search_runbooks(question)
    if not hits:
        return "[L1 RAG] No runbook found."
    top = hits[0]
    return f"[L1 RAG] From [{top['id']}] {top['title']}: {top['text']}"


def list_open_tickets(category: str | None = None) -> list[dict[str, Any]]:
    tickets = [t for t in TICKET_QUEUE if t["status"] == "open"]
    if category:
        tickets = [t for t in tickets if t["category"] == category.upper()]
    return tickets


def create_ticket(employee: str, subject: str, category: str = "NETWORK") -> dict[str, Any]:
    ticket = {
        "id": f"INC-{uuid.uuid4().hex[:4].upper()}",
        "employee": employee,
        "subject": subject,
        "status": "open",
        "category": category.upper(),
        "created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    TICKET_QUEUE.append(ticket)
    return ticket


def level_2_tool_loop(question: str, employee: str = "user@company.com") -> dict[str, Any]:
    """
    L2: Multi-step tool loop — search, check duplicates, optionally create ticket.
    Simulates what an LLM agent would orchestrate.
    """
    trace = []

    trace.append({"step": "reason", "detail": "User reports VPN issue — search runbooks first."})
    hits = search_runbooks("vpn connection")
    trace.append({"step": "act", "tool": "search_runbooks", "result": hits})

    trace.append({"step": "reason", "detail": "Check for existing open NETWORK tickets to avoid duplicates."})
    existing = list_open_tickets(category="NETWORK")
    trace.append({"step": "act", "tool": "list_open_tickets", "result": existing})

    runbook_answer = f"Follow [{hits[0]['id']}]: {hits[0]['text']}" if hits else "No runbook found."

    if not any(t["employee"] == employee for t in existing):
        trace.append({"step": "reason", "detail": "No ticket for this user — create one for tracking."})
        ticket = create_ticket(employee, "VPN connection issue from home", "NETWORK")
        trace.append({"step": "act", "tool": "create_ticket", "result": ticket})
        final = f"{runbook_answer} I also opened ticket {ticket['id']} for tracking."
    else:
        ticket_id = next(t["id"] for t in existing if t["employee"] == employee)
        final = f"{runbook_answer} You already have open ticket {ticket_id}."

    return {"level": "L2", "answer": final, "trace": trace}


def level_3_multi_role(question: str, employee: str = "user@company.com") -> dict[str, Any]:
    """
    L3: Planner delegates to triage + network specialist + ticket agent.
    Each role is a focused function — in production, each could be a separate LLM call.
    """
    roles_trace = []

    # Planner
    roles_trace.append({"role": "planner", "action": "Classify as NETWORK issue; delegate to triage."})

    # Triage specialist
    category = "NETWORK"
    roles_trace.append({"role": "triage", "action": f"Category={category}; hand off to network specialist."})

    # Network specialist
    hits = search_runbooks("vpn")
    roles_trace.append({"role": "network_specialist", "action": f"Retrieved {hits[0]['id'] if hits else 'none'}."})

    # Ticket agent
    l2 = level_2_tool_loop(question, employee)
    roles_trace.append({"role": "ticket_agent", "action": "Synced ticket queue."})

    return {
        "level": "L3",
        "answer": l2["answer"],
        "roles_trace": roles_trace,
        "tool_trace": l2["trace"],
    }


print("=" * 72)
print(f"USER: {USER_QUESTION}")
print("=" * 72)
print(level_0_static_prompt(USER_QUESTION)[:200], "...\n")
print(level_1_retrieve_and_answer(USER_QUESTION)[:200], "...\n")

l2_result = level_2_tool_loop(USER_QUESTION)
print("L2 trace:")
for step in l2_result["trace"]:
    print(f"  {step}")
print(f"L2 answer: {l2_result['answer'][:180]}...\n")

l3_result = level_3_multi_role(USER_QUESTION)
print("L3 role delegation:")
for step in l3_result["roles_trace"]:
    print(f"  [{step['role']}] {step['action']}")

---

## 3. Tool Use as Agency

**Agency** means the system's outputs can **change state** in the world and those changes **inform the next decision**. Tool use is the main mechanism for that in LLM systems.

```
 User: "I cannot connect to VPN"
   │
   ▼
 LLM ──► tool_call: search_runbooks(query="vpn connection")
   │
   ▼
 Your runtime ──► executes search_runbooks → returns runbook vpn-101
   │
   ▼
 observation appended to messages ──► LLM ──► "Try steps 1–3 from vpn-101..."
```

The model emits a **structured intent** (`name` + `arguments`). Your code validates and executes it. This separation is what makes tool use **safe, testable, and auditable**.

In [ ]:
def build_tool_call(name: str, arguments: dict[str, Any], call_id: str = "call_001") -> dict[str, Any]:
    """Shape mirrors OpenAI / Anthropic tool_calls — execution is always your job."""
    return {
        "id": call_id,
        "type": "function",
        "function": {
            "name": name,
            "arguments": json.dumps(arguments),
        },
    }


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "search_runbooks": search_runbooks,
    "list_open_tickets": list_open_tickets,
    "create_ticket": create_ticket,
}


def execute_tool_call(tool_call: dict[str, Any]) -> str:
    """Runtime dispatcher — this is where agency meets engineering."""
    fn_info = tool_call["function"]
    name = fn_info["name"]
    args = json.loads(fn_info["arguments"])

    if name not in TOOL_REGISTRY:
        return json.dumps({"error": f"Unknown tool: {name}"})

    result = TOOL_REGISTRY[name](**args)
    return json.dumps(result, default=str)


# Simulate an LLM emitting a tool call
simulated_call = build_tool_call("search_runbooks", {"query": "vpn home connection"})
print("LLM emitted:")
print(json.dumps(simulated_call, indent=2))
print("\nRuntime executed:")
print(execute_tool_call(simulated_call))

---

## 4. Memory — More Than the Context Window

Chat history in a single thread is only **working memory**. Real agents combine several memory types:

| Type | What it holds | IT support example |
|------|---------------|-------------------|
| **Working** | Current conversation messages | User describes VPN error; last tool result |
| **Episodic** | Past sessions / runs | "Last week we fixed Bob's VPN by clearing credentials" |
| **Semantic** | Facts, docs, embeddings | Runbooks vpn-101, pwd-102, access-103 |
| **Procedural** | How to act | Tool schemas, escalation policies, category codes |

The context window is a **cache**, not a database. Production agents persist episodic and semantic memory externally (vector DB, key-value store) and load only what is relevant per step.

In [ ]:
@dataclass
class AgentMemory:
    """In-memory model of the four memory types."""

    working: list[dict[str, str]] = field(default_factory=list)
    episodic: list[dict[str, str]] = field(default_factory=lambda: list(EPISODIC_MEMORY))
    semantic: list[dict[str, str]] = field(default_factory=lambda: list(RUNBOOKS))
    procedural: list[str] = field(default_factory=lambda: [
        "Always search runbooks before escalating.",
        "Never share admin credentials.",
        "Create a ticket if issue is unresolved after step 3.",
    ])

    def add_user_message(self, content: str) -> None:
        self.working.append({"role": "user", "content": content})

    def add_tool_result(self, tool_name: str, result: str) -> None:
        self.working.append({"role": "tool", "name": tool_name, "content": result})

    def recall_similar_episode(self, keyword: str) -> str | None:
        keyword = keyword.lower()
        for episode in self.episodic:
            if keyword in episode["summary"].lower():
                return episode["summary"]
        return None

    def snapshot(self) -> dict[str, Any]:
        return {
            "working_messages": len(self.working),
            "episodic_sessions": len(self.episodic),
            "semantic_docs": [d["id"] for d in self.semantic],
            "procedural_rules": len(self.procedural),
        }


memory = AgentMemory()
memory.add_user_message("VPN keeps disconnecting at home")
memory.add_tool_result("search_runbooks", json.dumps(search_runbooks("vpn")))

print("Memory snapshot:", memory.snapshot())
print("Similar past episode:", memory.recall_similar_episode("vpn"))

---

## 5. Planning — Explicit vs Implicit

**Implicit planning** — the model decides the next action at each step (classic ReAct). There is no separate plan artifact; reasoning is interleaved with tool calls.

**Explicit planning** — a planner produces a numbered list of steps first; an executor follows them one by one. Easier to audit; can be rigid if the world changes mid-run.

```
Implicit (ReAct):        think → act → observe → think → act → ...

Explicit (plan-execute): plan [step1, step2, step3] → execute step1 → execute step2 → ...
```

Many production agents use a hybrid: explicit plan for the first pass, implicit replanning when a tool fails.

In [ ]:
def implicit_react_trace(user_goal: str) -> list[str]:
    """Simulates interleaved thought / action / observation steps."""
    hits = search_runbooks("vpn")
    return [
        f"Thought: User goal is '{user_goal}'. I should check VPN runbooks.",
        "Action: search_runbooks(query='vpn')",
        f"Observation: found {hits[0]['id']} — {hits[0]['title']}",
        "Thought: I have enough to answer; no ticket needed for informational request.",
        f"Answer: {hits[0]['text'][:100]}...",
    ]


def explicit_plan(user_goal: str) -> list[str]:
    """Returns a fixed plan before any execution."""
    return [
        "1. Classify issue category (NETWORK / ACCESS / PASSWORD).",
        "2. Search runbooks for matching category.",
        "3. Check episodic memory for similar resolved cases.",
        "4. List open tickets for the user to avoid duplicates.",
        "5. Compose answer with runbook citations; create ticket if unresolved.",
    ]


def execute_explicit_plan(plan: list[str], user_goal: str) -> list[str]:
    """Walk through an explicit plan with real tool calls."""
    log = [f"PLAN: {len(plan)} steps"]
    for step in plan:
        log.append(f"Executing: {step}")
        if "Search runbooks" in step:
            log.append(f"  → {search_runbooks('vpn')[0]['id']}")
        if "episodic" in step:
            episode = AgentMemory().recall_similar_episode("vpn")
            log.append(f"  → {episode}")
    log.append(f"DONE: answered goal '{user_goal}'")
    return log


print("--- Implicit ReAct trace ---")
for line in implicit_react_trace("VPN issue from home"):
    print(line)

print("\n--- Explicit plan + execution ---")
plan = explicit_plan("VPN issue from home")
for line in execute_explicit_plan(plan, "VPN issue from home"):
    print(line)

---

## 6. Feedback — Closing the Loop

An agent improves within a run (and sometimes across runs) through **feedback**:

| Source | What it looks like | Agent response |
|--------|-------------------|----------------|
| **Tool error** | `{"error": "employee email required"}` | Retry with corrected arguments |
| **Validator** | JSON schema rejects malformed args | Ask model to fix the tool call |
| **Critic** | Rule or second pass: "missing runbook citation" | Add citation and regenerate |
| **Human (HITL)** | Operator approves before `create_ticket` | Pause until approved |

Without feedback, the agent treats every tool result as success — including silent failures.

In [ ]:
def create_ticket_validated(employee: str, subject: str, category: str = "NETWORK") -> dict[str, Any]:
    """Tool with validation — errors become feedback for the agent loop."""
    if not employee or "@" not in employee:
        return {"success": False, "error": "employee must be a valid email"}
    if len(subject.strip()) < 5:
        return {"success": False, "error": "subject too short"}
    ticket = create_ticket(employee, subject, category)
    return {"success": True, "ticket": ticket}


def run_with_feedback(tool_name: str, args: dict[str, Any], max_retries: int = 2) -> dict[str, Any]:
    """Simulate an agent loop that retries on validation errors."""
    trace = []
    for attempt in range(max_retries + 1):
        if tool_name == "create_ticket":
            result = create_ticket_validated(**args)
        else:
            result = {"success": True, "data": TOOL_REGISTRY[tool_name](**args)}

        trace.append({"attempt": attempt + 1, "args": args, "result": result})

        if result.get("success", True) and "error" not in result:
            return {"final": result, "trace": trace}

        # Feedback: fix args based on error message
        if "email" in result.get("error", ""):
            args["employee"] = "user@company.com"
        if "subject" in result.get("error", ""):
            args["subject"] = "VPN connection issue from home"

    return {"final": result, "trace": trace}


# First call fails (bad email), feedback loop fixes it
outcome = run_with_feedback(
    "create_ticket",
    {"employee": "not-an-email", "subject": "VPN", "category": "NETWORK"},
)
for step in outcome["trace"]:
    print(step)

---

## 7. The Augmented LLM — Capabilities Around the Model

Researchers often describe modern LLM applications as an **Augmented LLM**: the same core model surrounded by augmentations:

```
            ┌─────────────┐
            │     LLM     │
            └──────┬──────┘
     retrieval │ tools │ memory │ planning
               ▼      ▼      ▼        ▼
            [RAG] [APIs] [store] [task list]
```

| Augmentation | Purpose | In our scenario |
|--------------|---------|-----------------|
| **Retrieval** | Ground answers in fresh docs | `search_runbooks` |
| **Tools** | Act on external systems | `create_ticket`, `list_open_tickets` |
| **Memory** | Persist context beyond one window | `AgentMemory`, `EPISODIC_MEMORY` |
| **Planning** | Structure multi-step work | Explicit plan or ReAct trace |

An **agent** is an Augmented LLM **plus a control loop** that decides when to retrieve, call tools, read memory, replan, and stop. The augmentations are ingredients; the loop is the recipe.

---

## 8. Agency Ingredients — The 6-Point Checklist

Use this scorecard on any product feature. Score each criterion **yes/no**:

| # | Criterion | Question to ask |
|---|-----------|-----------------|
| 1 | **Goal** | Is there a task beyond answering one question? |
| 2 | **Loop** | Can it take multiple steps before finishing? |
| 3 | **Tools** | Can it affect external systems or data? |
| 4 | **State** | Do messages and observations persist across steps? |
| 5 | **Stop rule** | Is it clear what ends the run? |
| 6 | **Guardrails** | Is there a defined set of forbidden actions? |

**Scoring guide:**
- **0–1 yes** → chatbot / static prompt
- **2–3 yes** → augmented chat (RAG, single tool call)
- **4–6 yes** → agent

This is a design tool, not a law. A feature with score 3 might still be the right choice if it is simpler and cheaper.

In [ ]:
def agent_scorecard(features: dict[str, bool]) -> tuple[int, str]:
    score = sum(features.values())
    if score >= 4:
        label = "agent"
    elif score >= 2:
        label = "augmented chat"
    else:
        label = "chatbot"
    return score, label


CANDIDATES = {
    "Static VPN FAQ page": {
        "goal": False, "loop": False, "tools": False,
        "state": False, "stop_rule": True, "guardrails": True,
    },
    "Runbook search widget (RAG)": {
        "goal": True, "loop": False, "tools": False,
        "state": False, "stop_rule": True, "guardrails": True,
    },
    "IT ticket assistant (tool loop)": {
        "goal": True, "loop": True, "tools": True,
        "state": True, "stop_rule": True, "guardrails": True,
    },
    "Autonomous infra repair bot": {
        "goal": True, "loop": True, "tools": True,
        "state": True, "stop_rule": False, "guardrails": False,
    },
}

print(f"{'Feature':<35} {'Score':>5}  Classification")
print("-" * 60)
for name, feats in CANDIDATES.items():
    score, label = agent_scorecard(feats)
    print(f"{name:<35} {score}/6   {label}")

---

## 9. ReAct — Reason and Act Interleaved

**ReAct** (Reason + Act) is a prompting and control pattern where the model alternates between **reasoning traces** and **tool actions**. It is one of the most common ways to implement implicit planning.

A typical ReAct step looks like:

```
Thought: The user cannot connect to VPN. I should look up the VPN runbook.
Action: search_runbooks
Action Input: {"query": "vpn connection"}
Observation: [{"id": "vpn-101", ...}]
```

The runtime parses the `Action` line, runs the tool, injects the `Observation`, and calls the model again.

In [ ]:
REACT_STEP_TEMPLATE = """Thought: {thought}
Action: {action}
Action Input: {action_input}
Observation: {observation}"""


@dataclass
class ReActAgent:
    """Minimal ReAct loop — planner rules stand in for an LLM."""

    messages: list[dict[str, Any]] = field(default_factory=list)
    max_steps: int = 4

    def run(self, user_goal: str) -> str:
        self.messages = [{"role": "user", "content": user_goal}]

        # Step 1: search runbooks
        thought = "User reports VPN trouble. Search runbooks first."
        action, args = "search_runbooks", {"query": "vpn"}
        observation = execute_tool_call(build_tool_call(action, args))
        self.messages.append({"role": "assistant", "content": REACT_STEP_TEMPLATE.format(
            thought=thought, action=action, action_input=json.dumps(args), observation=observation[:80]
        )})

        # Step 2: check tickets
        thought = "Check if user already has an open NETWORK ticket."
        action, args = "list_open_tickets", {"category": "NETWORK"}
        observation = execute_tool_call(build_tool_call(action, args, "call_002"))
        self.messages.append({"role": "assistant", "content": REACT_STEP_TEMPLATE.format(
            thought=thought, action=action, action_input=json.dumps(args), observation=observation[:80]
        )})

        hits = search_runbooks("vpn")
        answer = f"Try runbook [{hits[0]['id']}]: {hits[0]['text'][:120]}..."
        self.messages.append({"role": "assistant", "content": f"Final Answer: {answer}"})
        return answer


react_agent = ReActAgent()
print(react_agent.run("VPN will not connect from my home network"))
print(f"\nReAct trace: {len(react_agent.messages)} assistant steps recorded")

---

## 10. Structured Outputs vs Free Text

Early demos used prompts like *"respond with JSON containing the tool name"* — fragile and unreliable. Modern agent stacks use **schema-defined tool calls** returned by the API.

| Approach | Reliability | When to use |
|----------|-------------|-------------|
| **API tool schemas** | High — model trained for function calling | Production agents |
| **Pydantic validation** | High — catch bad args before execution | Every tool boundary |
| **Regex on free text** | Low — breaks on formatting drift | Never in production |

Always validate tool arguments **in your runtime** even when the API returns structured JSON. The model can still hallucinate invalid enum values or missing required fields.

In [ ]:
VALID_CATEGORIES = {"NETWORK", "ACCESS", "PASSWORD", "OTHER"}


def validate_ticket_args(args: dict[str, Any]) -> tuple[bool, str]:
    """Pydantic-style validation without extra dependencies."""
    if "employee" not in args or "@" not in str(args["employee"]):
        return False, "employee must be a valid email"
    if "subject" not in args or len(str(args["subject"]).strip()) < 5:
        return False, "subject must be at least 5 characters"
    category = str(args.get("category", "OTHER")).upper()
    if category not in VALID_CATEGORIES:
        return False, f"category must be one of {sorted(VALID_CATEGORIES)}"
    return True, "ok"


TEST_ARGS = [
    {"employee": "alice@company.com", "subject": "VPN drops at home", "category": "NETWORK"},
    {"employee": "alice", "subject": "VPN", "category": "NETWORK"},
    {"employee": "alice@company.com", "subject": "VPN issue", "category": "INVALID"},
]

for args in TEST_ARGS:
    ok, msg = validate_ticket_args(args)
    status = "ALLOW" if ok else f"BLOCK ({msg})"
    print(f"{status}: {args}")

---

## 11. Putting It Together — A Checklist-Complete Agent

The class below satisfies all six checklist criteria for the IT Support scenario. Compare it to L0/L1 implementations earlier — the difference is the **runtime loop**, not a smarter model.

In [ ]:
FORBIDDEN_ACTIONS = {"delete_all_tickets", "share_admin_password", "disable_mfa"}


@dataclass
class ITSupportAgent:
    """Checklist-complete agent: goal, loop, tools, state, stop rule, guardrails."""

    memory: AgentMemory = field(default_factory=AgentMemory)
    max_steps: int = 5
    allowed_tools: set[str] = field(default_factory=lambda: set(TOOL_REGISTRY.keys()))

    def run(self, user_message: str, employee: str = "user@company.com") -> dict[str, Any]:
        self.memory.add_user_message(user_message)
        trace: list[dict[str, Any]] = []

        # Guardrail: block dangerous intents early
        if any(word in user_message.lower() for word in ("admin password", "disable mfa", "delete all")):
            return {
                "answer": "Request blocked by safety policy.",
                "trace": [{"guardrail": "forbidden intent detected"}],
            }

        steps = [
            ("search_runbooks", {"query": "vpn" if "vpn" in user_message.lower() else user_message}),
            ("list_open_tickets", {"category": "NETWORK"}),
        ]

        runbook_text = ""
        for i, (tool_name, args) in enumerate(steps):
            if i >= self.max_steps:
                break
            if tool_name not in self.allowed_tools or tool_name in FORBIDDEN_ACTIONS:
                trace.append({"error": f"tool {tool_name} not allowed"})
                break

            observation = execute_tool_call(build_tool_call(tool_name, args, f"call_{i}"))
            self.memory.add_tool_result(tool_name, observation)
            trace.append({"tool": tool_name, "args": args, "observation": observation[:100]})

            if tool_name == "search_runbooks":
                hits = json.loads(observation)
                if hits:
                    runbook_text = f"[{hits[0]['id']}] {hits[0]['text']}"

        # Stop rule: create ticket only if VPN-related and no existing user ticket
        if "vpn" in user_message.lower():
            open_tickets = list_open_tickets("NETWORK")
            if not any(t["employee"] == employee for t in open_tickets):
                result = create_ticket_validated(employee, "VPN connection issue", "NETWORK")
                if result.get("success"):
                    ticket_id = result["ticket"]["id"]
                    answer = f"{runbook_text} Opened tracking ticket {ticket_id}."
                else:
                    answer = f"{runbook_text} (Could not open ticket: {result.get('error')})"
            else:
                existing_id = next(t["id"] for t in open_tickets if t["employee"] == employee)
                answer = f"{runbook_text} Existing ticket: {existing_id}."
        else:
            answer = runbook_text or "No runbook match found."

        return {"answer": answer, "trace": trace, "memory": self.memory.snapshot()}


agent = ITSupportAgent(max_steps=3)
result = agent.run("I cannot connect to VPN from home")
print("ANSWER:", result["answer"][:200], "...")
print("TRACE:", json.dumps(result["trace"], indent=2))
print("MEMORY:", result["memory"])

---

## 12. LLM Limits That Shape Agency

Even a perfect agent loop is constrained by the model underneath:

| Limit | Impact on agents | Mitigation |
|-------|------------------|------------|
| **Hallucination** | Wrong tool names or arguments | Schema validation, retries, smaller tool sets |
| **Context size** | Early steps forgotten | Summarization, external memory store |
| **Latency** | Slow multi-step loops | Parallel tools, smaller models for routing |
| **Nondeterminism** | Flaky runs across retries | Low temperature for tool calls, eval harnesses |
| **No true state** | Model does not "remember" between calls unless you pass it | Explicit message history + memory retrieval |

Agency is achieved in **software around the model**, not by assuming the model is reliable on its own.

---

## 13. When Tool Use Alone Is Not Enough

A single agent with many tools works until:

- **Prompts become unmaintainable** — one system message tries to cover triage, networking, access, and tone.
- **Roles conflict** — security reviewer and implementer need opposite instructions.
- **Subtasks are parallel** — researching a runbook while drafting a user email.
- **Specialists need different models** — cheap router + capable reasoner.

That is when teams split into **multiple agents** with a planner or supervisor. The perceive–reason–act loop still applies to each agent; you add **coordination** on top (as in the L3 demo earlier).

Start with one agent. Split only when the checklist stays green but maintainability degrades.

---

## 14. Optional — Live LLM Perspective

Set `USE_LIVE_LLM = True` with a valid API key to ask a real model whether tool calling counts as agency. The agent code above does not change — only the **planner** would be swapped from rules to an API call.

In [ ]:
USE_LIVE_LLM = False

OFFLINE_ANSWER = (
    "Yes — tool calling is agency when your runtime executes the call and feeds the "
    "observation back into the loop. The model proposes; the agent runtime disposes."
)

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": "In one sentence: does tool calling give an LLM agency?",
            }],
            max_tokens=60,
        )
        print(resp.choices[0].message.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print(OFFLINE_ANSWER)

---

## 15. Common Mistakes

| Mistake | Why it hurts | Better approach |
|---------|--------------|------------------|
| Equating streaming with agency | Streaming is UX; agency is loops + tools | Check the 6-point checklist |
| One giant system prompt | Context bloat; conflicting instructions | Split roles or use memory retrieval |
| No observation formatting | Model misreads raw API dumps | Standardize tool result JSON |
| Ignoring stop conditions | Runaway cost and infinite loops | `max_steps`, budgets, clear done states |
| Calling any RAG app an "agent" | Retrieval alone is L1, not a loop | Reserve "agent" for multi-step tool loops |
| Trusting model output as executed | User thinks action happened | Show tool trace and confirmation |

---

## 16. Check Your Understanding

1. What is the difference between an LLM **predictor** and an LLM **agent**?
2. Describe L0, L1, L2, and L3 on the autonomy spectrum using the VPN example.
3. Why does tool use count as **agency** only when a runtime executes the call?
4. Name the four memory types and give one IT-support example for each.
5. What is the difference between **implicit** (ReAct) and **explicit** planning?
6. Which two features in the scorecard table are "agents" and which are not? Why?
7. What does an **Augmented LLM** add around the core model, and what does the **agent loop** add on top?

---

## 17. Summary

An **LLM agent** is not a different model — it is a **predictor plus a runtime** that provides tools, memory, planning, feedback, and stop rules.

**Key takeaways:**

- **Autonomy is a spectrum** from static prompts (L0) to multi-role systems (L3). Pick the lowest level that solves the problem.
- **Tool use is agency** when structured calls change environment state and observations flow back into the loop.
- **Memory** spans working, episodic, semantic, and procedural layers — the context window alone is not enough.
- **Planning** can be implicit (ReAct) or explicit (plan-then-execute); many systems combine both.
- **Feedback** from tool errors, validators, critics, and humans keeps the loop honest.
- The **6-point checklist** (goal, loop, tools, state, stop rule, guardrails) classifies any feature as chatbot, augmented chat, or agent.
- An **Augmented LLM** names the capabilities around the model; the **agent loop** orchestrates them over time.

You can now look at any AI product feature and ask: *Where is it on the autonomy spectrum, and does it pass the checklist?*